# Deep Generative Models - CA1
## Variational Autoencoders (VAE)
### University of Tehran - Electrical and Computer Engineering

**Course:** Deep Generative Models  
**Instructor:** Dr. Mostafa Tavasoli Pour

---

This notebook contains the implementation for CA1 homework covering:
1. **Question 1:** Probabilistic Graphical Models (Bayesian and Markov Networks)
2. **Question 2:** Variational Autoencoders implementation on dSprites dataset

# Question 1: Probabilistic Graphical Models

## Part 1: Bayesian Network - Disease Model

### Sub-part 1: Draw Bayesian Network

Based on the problem description, we have the following variables and their relationships:
- **M**: Immune system strength
- **S**: Season
- **I**: Disease severity  
- **F**: Financial capability
- **T**: Treatment type (expensive vs cheap)
- **D**: Probability of death

**Relationships:**
- Season (S) and Immune system (M) affect Disease severity (I)
- Disease severity (I) affects Probability of death (D)
- Disease severity (I) and Financial capability (F) affect Treatment (T)
- Treatment (T) affects Probability of death (D)

---

## Overview of Probabilistic Graphical Models

Probabilistic Graphical Models (PGMs) provide a framework for representing complex probability distributions using graphs. They come in two main types:

**Bayesian Networks (Directed)**:
- Nodes represent random variables
- Directed edges represent conditional dependencies
- Joint distribution factorizes as: $P(X_1, ..., X_n) = \prod_i P(X_i | Parents(X_i))$

**Markov Networks (Undirected)**:
- Nodes represent random variables
- Undirected edges represent dependencies
- Joint distribution uses clique potentials: $P(X) = \frac{1}{Z} \prod_c \phi_c(X_c)$

### Implementation Tasks
1. Draw and analyze a disease model Bayesian network
2. Compute conditional independence statements
3. Analyze a given network structure
4. Convert to Markov network and find maximal cliques

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

def draw_bayesian_network():
    fig, ax = plt.subplots(1, 1, figsize=(12, 8))
    
    G = nx.DiGraph()
    
    nodes = ['M', 'S', 'I', 'F', 'T', 'D']
    node_labels = {
        'M': 'Immune System\n(M)',
        'S': 'Season\n(S)',
        'I': 'Disease Severity\n(I)',
        'F': 'Financial Capability\n(F)',
        'T': 'Treatment Type\n(T)',
        'D': 'Death Probability\n(D)'
    }
    
    edges = [
        ('M', 'I'),
        ('S', 'I'),
        ('I', 'D'),
        ('I', 'T'),
        ('F', 'T'),
        ('T', 'D'),
    ]
    
    G.add_edges_from(edges)
    
    pos = {
        'M': (0, 2),
        'S': (2, 2),
        'I': (1, 1),
        'F': (3, 1),
        'T': (2, 0),
        'D': (1, -1)
    }
    
    nx.draw_networkx_nodes(G, pos, node_color='lightblue', 
                          node_size=3000, alpha=0.9, ax=ax)
    nx.draw_networkx_labels(G, pos, node_labels, font_size=9, 
                           font_weight='bold', ax=ax)
    nx.draw_networkx_edges(G, pos, edge_color='gray', 
                          arrows=True, arrowsize=20, 
                          arrowstyle='->', width=2, ax=ax)
    
    ax.set_title('Bayesian Network: Disease Model', fontsize=14, fontweight='bold')
    ax.axis('off')
    plt.tight_layout()
    plt.savefig('bayesian_network.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    return G, edges

G, edges = draw_bayesian_network()
print("Bayesian Network Structure:")
print(f"Nodes: {list(G.nodes())}")
print(f"Edges: {edges}")


### Implementation: Bayesian Network Visualization

This code creates a visual representation of the disease model with:
- **NetworkX**: For graph structure and layout
- **Matplotlib**: For rendering the network
- **Custom positioning**: Hierarchical layout showing causal flow

The network captures medical decision-making:
- Environmental factors (Season, Immune system) → Disease severity
- Disease severity + Financial capability → Treatment choice
- Disease severity + Treatment → Death probability

### Sub-part 2: Joint Probability Distribution

The joint probability distribution for the Bayesian Network can be written as:

$$P(M, S, I, F, T, D) = P(M) \cdot P(S) \cdot P(F) \cdot P(I|M, S) \cdot P(T|I, F) \cdot P(D|I, T)$$

This factorization follows from the conditional independence structure of the Bayesian network.

### Sub-part 3: Conditional Independence Statements

Let's analyze each statement:

In [ ]:
def analyze_conditional_independence():
    statements = {
        'a': {
            'statement': 'F ⊥ D',
            'description': 'F is independent of D (unconditionally)',
            'answer': False,
            'reasoning': 'F and D are NOT independent because there is an active path F → T → D. '
                        'Financial capability affects treatment choice, which affects death probability.'
        },
        'b': {
            'statement': 'S ⊥ D | I',
            'description': 'S is independent of D given I',
            'answer': True,
            'reasoning': 'Given I (disease severity), S (season) is independent of D (death probability). '
                        'Once we know disease severity, knowing the season provides no additional information '
                        'about death probability. The path S → I → D is blocked by observing I.'
        },
        'c': {
            'statement': 'M ⊥ F',
            'description': 'M is independent of F (unconditionally)',
            'answer': True,
            'reasoning': 'M (immune system) and F (financial capability) are independent. '
                        'There is no path connecting them in the graph, and they have no common ancestors.'
        },
        'd': {
            'statement': 'M ⊥ F | T',
            'description': 'M is independent of F given T',
            'answer': False,
            'reasoning': 'Given T (treatment type), M and F become DEPENDENT. This is a V-structure (collider): '
                        'M → I ← S and I → T ← F. Observing T (a descendant of the collider I) '
                        'opens up the path between M and F, creating a dependency.'
        },
        'e': {
            'statement': 'M ⊥ T | {D, I}',
            'description': 'M is independent of T given both D and I',
            'answer': True,
            'reasoning': 'Given both I and D, M is independent of T. '
                        'The path M → I → T is blocked by observing I. '
                        'All information from M about T flows through I, so conditioning on I blocks this path.'
        }
    }
    
    print("="*80)
    print("CONDITIONAL INDEPENDENCE ANALYSIS")
    print("="*80)
    
    for key, data in statements.items():
        print(f"\n{key}. {data['statement']}")
        print(f"   Description: {data['description']}")
        print(f"   Answer: {'TRUE' if data['answer'] else 'FALSE'}")
        print(f"   Reasoning: {data['reasoning']}")
    
    return statements

independence_analysis = analyze_conditional_independence()


### Conditional Independence Analysis

This function analyzes conditional independence using **d-separation** rules:

**Key Concepts**:
1. **Path blocking**: Conditioning on a variable can block information flow
2. **V-structure (collider)**: X → Z ← Y creates dependence when Z is observed
3. **Serial connection**: X → Y → Z blocks path when Y is observed
4. **Diverging connection**: X ← Y → Z blocks path when Y is observed

Each statement is evaluated by tracing paths in the graph and checking whether they're active or blocked given the conditioning set.

## Part 2: Given Bayesian Network Analysis

Graph structure:
```
    C
    |
    O - A
    |   |
    S - T - B
        |
        M
```

In [ ]:
def draw_given_bayesian_network():
    fig, ax = plt.subplots(1, 1, figsize=(10, 8))
    
    G = nx.DiGraph()
    
    edges = [
        ('C', 'O'),
        ('O', 'A'),
        ('O', 'S'),
        ('A', 'T'),
        ('S', 'T'),
        ('T', 'B'),
        ('T', 'M')
    ]
    
    G.add_edges_from(edges)
    
    pos = {
        'C': (1, 3),
        'O': (1, 2),
        'A': (2, 1.5),
        'S': (0, 1),
        'T': (1, 0.5),
        'B': (2, -0.5),
        'M': (0, -0.5)
    }
    
    nx.draw_networkx_nodes(G, pos, node_color='lightgreen', 
                          node_size=2000, alpha=0.9, ax=ax)
    nx.draw_networkx_labels(G, pos, font_size=12, font_weight='bold', ax=ax)
    nx.draw_networkx_edges(G, pos, edge_color='gray', 
                          arrows=True, arrowsize=20, 
                          arrowstyle='->', width=2, ax=ax)
    
    ax.set_title('Given Bayesian Network', fontsize=14, fontweight='bold')
    ax.axis('off')
    plt.tight_layout()
    plt.savefig('given_bayesian_network.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    return G

print("Sub-part 1: Joint Probability Distribution")
print("="*60)
print("P(C, O, A, S, T, B, M) = P(C) · P(O|C) · P(S|O) · P(A|O) · P(T|A,S) · P(B|T) · P(M|T)")
print()

print("Sub-part 2: Markov Blanket of T")
print("="*60)
print("Markov Blanket of T = {A, S, B, M}")
print("This includes:")
print("- Parents of T: {A, S}")
print("- Children of T: {B, M}")
print("- Co-parents (other parents of T's children): {} (none in this case)")
print()

G = draw_given_bayesian_network()


### Given Network Analysis

This section analyzes a predefined Bayesian network structure with the following tasks:

**Sub-part 1**: Factorize the joint probability distribution following the chain rule for Bayesian networks.

**Sub-part 2**: Find the **Markov blanket** of node T:
- Parents: nodes with edges pointing to T
- Children: nodes that T points to
- Co-parents: other parents of T's children

The Markov blanket is the minimal set of variables that makes T independent of all other variables in the network.

### Sub-part 3: Markov Network Analysis

Now analyzing the undirected Markov network version of the same graph.

In [ ]:
def draw_markov_network():
    fig, ax = plt.subplots(1, 1, figsize=(10, 8))
    
    G = nx.Graph()
    
    edges = [
        ('C', 'O'),
        ('O', 'A'),
        ('O', 'S'),
        ('A', 'T'),
        ('S', 'T'),
        ('T', 'B'),
        ('T', 'M')
    ]
    
    G.add_edges_from(edges)
    
    pos = {
        'C': (1, 3),
        'O': (1, 2),
        'A': (2, 1.5),
        'S': (0, 1),
        'T': (1, 0.5),
        'B': (2, -0.5),
        'M': (0, -0.5)
    }
    
    nx.draw_networkx_nodes(G, pos, node_color='lightcoral', 
                          node_size=2000, alpha=0.9, ax=ax)
    nx.draw_networkx_labels(G, pos, font_size=12, font_weight='bold', ax=ax)
    nx.draw_networkx_edges(G, pos, edge_color='gray', width=2, ax=ax)
    
    ax.set_title('Markov Network (Undirected)', fontsize=14, fontweight='bold')
    ax.axis('off')
    plt.tight_layout()
    plt.savefig('markov_network.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    return G

def is_perfect_imap(bayesian_edges, markov_edges):
    print("\nSub-part 3: Is this a Perfect I-Map?")
    print("="*60)
    print("Answer: NO")
    print("\nReasoning:")
    print("A perfect I-map means the Markov network can represent exactly")
    print("the same independence structure as the Bayesian network.")
    print("\nThe Bayesian network has conditional independences that cannot")
    print("be represented in the Markov network. For example:")
    print("- In Bayesian: C ⊥ {S,A,T,B,M} | O (C is independent of others given O)")
    print("- In Markov: This independence is lost when we moralize the graph")
    print("\nTherefore, the Markov network is NOT a perfect I-map.")
    
def is_chordal(G):
    print("\nSub-part 4: Is the graph chordal?")
    print("="*60)
    
    try:
        is_chordal_graph = nx.is_chordal(G)
        print(f"Answer: {'YES' if is_chordal_graph else 'NO'}")
        
        if is_chordal_graph:
            print("\nThe graph is chordal. Every cycle of length ≥ 4 has a chord.")
        else:
            print("\nThe graph is NOT chordal.")
            print("There exists at least one cycle of length ≥ 4 without a chord.")
        
        return is_chordal_graph
    except:
        print("Chordality check requires further analysis")
        return None

def find_maximal_cliques(G):
    print("\nSub-part 5: Maximal Cliques")
    print("="*60)
    
    cliques = list(nx.find_cliques(G))
    
    print(f"Number of maximal cliques: {len(cliques)}")
    for i, clique in enumerate(cliques, 1):
        print(f"Clique {i}: {{{', '.join(sorted(clique))}}}")
    
    print("\nJoint Probability based on maximal cliques:")
    print("P(C,O,A,S,T,B,M) = (1/Z) × ", end="")
    clique_potentials = [f"φ({{{','.join(sorted(c))}}})" for c in cliques]
    print(" × ".join(clique_potentials))
    
    return cliques

print("MARKOV NETWORK ANALYSIS")
print("="*80)

G_markov = draw_markov_network()

bayesian_edges = [
    ('C', 'O'), ('O', 'A'), ('O', 'S'),
    ('A', 'T'), ('S', 'T'), ('T', 'B'), ('T', 'M')
]

is_perfect_imap(bayesian_edges, list(G_markov.edges()))
is_chordal(G_markov)
cliques = find_maximal_cliques(G_markov)


### Markov Network Properties

Converting to an undirected Markov network involves **moralization**:
1. Add undirected edges for all directed edges
2. "Marry" parents: add edges between parents of common children
3. Drop edge directions

**Analysis Tasks**:
- **Perfect I-map**: Check if independence properties are preserved
- **Chordality**: Every cycle of length ≥ 4 must have a chord (shortcut edge)
- **Maximal cliques**: Find all maximal fully-connected subgraphs
- **Clique factorization**: Express joint probability as product of clique potentials

### Sub-part 6: Removing a Variable in Bayesian vs Markov Networks

**Question**: Compare the effect of marginalizing out variable C in Bayesian vs Markov networks.

**Key Difference**:

**In Bayesian Network**:
- To marginalize out C, we only need to remove P(C) from the factorization
- Joint becomes: P(O,A,S,T,B,M) = P(O) · P(S|O) · P(A|O) · P(T|A,S) · P(B|T) · P(M|T)
- Simple: just drop the factor involving C

**In Markov Network**:
- Need to marginalize out C from all cliques containing it
- Must compute: φ_new(clique \ {C}) = ∫ φ(clique) dC
- For continuous C: ∫_{-∞}^{∞} φ(C, ...) dC
- Result depends on whether ∫ φ(C) dC = 1

**Normalization Requirement**:
The statement says we must have ∫_{-∞}^{∞} φ(C) dC = 1 for proper marginalization.

**Why?**
- If ∫ φ(C) dC ≠ 1, the normalized distribution changes after marginalization
- The partition function Z changes unpredictably
- We lose the proper normalization of the remaining variables

In [ ]:
# Demonstrate the difference in marginalization
print("Marginalizing out C: Bayesian vs Markov")
print("="*80)

print("\nBayesian Network:")
print("  Original: P(C,O,A,S,T,B,M) = P(C)·P(O|C)·P(S|O)·P(A|O)·P(T|A,S)·P(B|T)·P(M|T)")
print("  Remove C: P(O,A,S,T,B,M) = Σ_C [P(C)·P(O|C)]·P(S|O)·P(A|O)·P(T|A,S)·P(B|T)·P(M|T)")
print("           = P(O)·P(S|O)·P(A|O)·P(T|A,S)·P(B|T)·P(M|T)")
print("  Operation: Simply drop P(C) and marginalize in P(O|C)")

print("\nMarkov Network:")
print("  Original: P(X) = (1/Z) × Π φ_clique(X_clique)")
print("  Remove C: P(X\\C) = (1/Z') × Π φ_new(X_clique\\C)")
print("           where φ_new = ∫ φ(X_clique) dC")
print("  ")
print("  Requirement: ∫ φ(C) dC = 1 ensures Z' is properly normalized")

print("\n" + "="*80)
print("PROOF for Markov Networks:")
print("="*80)
print("""
Consider removing C from a clique {C, O}:

Original: P(C, O, ...) = (1/Z) × φ(C, O) × φ(other cliques)

Marginalizing C:
    P(O, ...) = ∫ P(C, O, ...) dC
              = (1/Z) × [∫ φ(C, O) dC] × φ(other cliques)
              = (1/Z) × φ_new(O) × φ(other cliques)

For proper normalization:
    If ∫ φ(C) dC = 1, then:
    Z' = Z / ∫ φ(C) dC = Z
    
    So we can write:
    P(O, ...) = (1/Z') × φ_new(O) × φ(other cliques)
    
If ∫ φ(C) dC ≠ 1, then Z' ≠ Z and we must recompute partition function!

In Bayesian networks, we don't have this issue because:
- Factors are already conditional probabilities (sum/integrate to 1)
- No partition function to worry about
""")

# Question 2: Variational Autoencoders (VAE)

## Part 1: Theoretical Questions

### Sub-part 1: Why don't we directly maximize log-likelihood?

The VAE loss function is:
$$\mathbb{E}_{q(z|x)}[\log p(x|z)] - D_{KL}(q(z|x) || p(z))$$

**Answer:**
We don't directly maximize $\log p(x)$ because:

1. **Intractability**: The true posterior $p(z|x) = \frac{p(x|z)p(z)}{p(x)}$ requires computing $p(x) = \int p(x|z)p(z)dz$, which is intractable for complex models.

2. **ELBO as Lower Bound**: Instead, we maximize the Evidence Lower BOund (ELBO):
   $$\log p(x) \geq \mathbb{E}_{q(z|x)}[\log p(x|z)] - D_{KL}(q(z|x) || p(z))$$

3. **Two Terms**:
   - **Reconstruction term** $\mathbb{E}_{q(z|x)}[\log p(x|z)]$: Encourages the decoder to reconstruct data
   - **KL term** $D_{KL}(q(z|x) || p(z))$: Regularizes the latent space to match the prior

By maximizing ELBO, we indirectly maximize $\log p(x)$ while keeping the problem tractable.

In [ ]:
def draw_seven_node_markov_network():
    """
    Draw the seven-node Markov network from Question 1 Part 3
    Graph structure:
           E
           |
       B - D - G
       |   |
       A - C - F
    """
    fig, ax = plt.subplots(1, 1, figsize=(12, 10))
    
    G = nx.Graph()
    
    # Define edges based on the structure
    edges = [
        ('A', 'B'),
        ('A', 'C'),
        ('B', 'D'),
        ('C', 'D'),
        ('C', 'F'),
        ('D', 'E'),
        ('D', 'G')
    ]
    
    G.add_edges_from(edges)
    
    # Position nodes for clear visualization
    pos = {
        'A': (0, 0),
        'B': (0, 2),
        'C': (2, 0),
        'D': (2, 2),
        'E': (2, 4),
        'F': (4, 0),
        'G': (4, 2)
    }
    
    # Draw network
    nx.draw_networkx_nodes(G, pos, node_color='lightblue', 
                          node_size=3000, alpha=0.9, ax=ax)
    nx.draw_networkx_labels(G, pos, font_size=16, font_weight='bold', ax=ax)
    nx.draw_networkx_edges(G, pos, edge_color='gray', width=3, ax=ax)
    
    ax.set_title('Seven-Node Markov Network (Question 1 Part 3)', 
                fontsize=16, fontweight='bold')
    ax.axis('off')
    plt.tight_layout()
    plt.savefig('seven_node_markov_network.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    return G, pos


def analyze_seven_node_network():
    """Complete analysis of the seven-node Markov network"""
    
    print("="*80)
    print("QUESTION 1 - PART 3: Seven-Node Markov Network Analysis")
    print("="*80)
    
    # Draw the network
    G, pos = draw_seven_node_markov_network()
    
    # Sub-part 1: Find maximal cliques
    print("\n" + "="*80)
    print("SUB-PART 1: Maximal Cliques")
    print("="*80)
    
    cliques = list(nx.find_cliques(G))
    maximal_cliques = [tuple(sorted(c)) for c in cliques]
    
    print(f"\nNumber of maximal cliques: {len(maximal_cliques)}")
    for i, clique in enumerate(sorted(maximal_cliques), 1):
        print(f"  Clique {i}: {{{', '.join(clique)}}}")
    
    print("\nJoint Probability Distribution:")
    print("  P(A,B,C,D,E,F,G) = (1/Z) × ", end="")
    clique_potentials = [f"φ({{{','.join(c)}}})" for c in sorted(maximal_cliques)]
    print(" × ".join(clique_potentials))
    
    # Sub-part 2: Conditional independence statements
    print("\n" + "="*80)
    print("SUB-PART 2: Conditional Independence Analysis")
    print("="*80)
    
    statements = {
        'a': {
            'statement': 'G ⊥ A',
            'answer': False,
            'path': 'G - D - B - A or G - D - C - A',
            'reasoning': 'G and A are NOT independent. There are active paths connecting them '
                        'through D: G-D-B-A and G-D-C-A. These paths are not blocked.'
        },
        'b': {
            'statement': 'F ⊥ A | {D, C}',
            'answer': True,
            'path': 'F - C - (D, A, B)',
            'reasoning': 'Given both D and C, F is independent of A. All paths from F to A '
                        'must go through C (F-C-A and F-C-D-B-A). Conditioning on C blocks '
                        'the direct path, and conditioning on D blocks the other path through D.'
        },
        'c': {
            'statement': 'G ⊥ C | E',
            'answer': False,
            'path': 'G - D - C',
            'reasoning': 'Given E, G and C are NOT independent. There is an active path G-D-C '
                        'that is not blocked by E. E only blocks paths going through E itself, '
                        'but G-D-C does not pass through E.'
        },
        'd': {
            'statement': 'P(A|B,C) = P(A|B,C,E)',
            'answer': True,
            'path': 'A - (B,C) - D - E',
            'reasoning': 'TRUE. Given B and C, A is independent of E. All paths from A to E '
                        'must go through either B or C (or both via D). Since we condition on '
                        'both B and C, these paths are blocked, making A ⊥ E | {B,C}.'
        }
    }
    
    for key, data in statements.items():
        print(f"\n{key}. {data['statement']}")
        print(f"   Answer: {'TRUE' if data['answer'] else 'FALSE'}")
        print(f"   Relevant path: {data['path']}")
        print(f"   Reasoning: {data['reasoning']}")
    
    # Sub-part 3: Effect of changing potential function
    print("\n" + "="*80)
    print("SUB-PART 3: Effect of Setting φ(E,G) = 5")
    print("="*80)
    
    print("\nOriginal distribution:")
    print("  P(A,B,C,D,E,F,G) = (1/Z) × φ(A,B) × φ(A,C) × φ(B,D) × φ(C,D) × φ(C,F) × φ(D,E) × φ(D,G)")
    
    print("\nIf we set φ(E,G) = 5:")
    print("  Problem: There is NO edge between E and G in the graph!")
    print("  The cliques are: {A,B}, {A,C}, {B,D}, {C,D}, {C,F}, {D,E}, {D,G}")
    print("  {E,G} is NOT a clique, so φ(E,G) is not a valid potential function.")
    
    print("\nWhat changes:")
    print("  1. If we ADD this potential anyway (treating it as a new factor):")
    print("     P'(X) = (1/Z') × [original potentials] × φ(E,G)")
    print("  ")
    print("  2. Effect:")
    print("     - This creates a dependency between E and G")
    print("     - Changes the graph structure (adds edge E-G)")
    print("     - Changes the independence structure")
    print("     - Z' ≠ Z (partition function changes)")
    
    print("\n  3. Setting φ(E,G) = 5 uniformly:")
    print("     - If φ(E,G) = 5 for all E,G values, it's just a constant")
    print("     - Can be absorbed into Z': Z' = 5 × Z")
    print("     - Doesn't change conditional probabilities")
    print("     - P'(X) = P(X) (distributions remain the same after normalization)")
    
    return G, maximal_cliques, statements


# Run the analysis
G, cliques, statements = analyze_seven_node_network()

---

## Question 2: Variational Autoencoders

### Overview of VAE

Variational Autoencoders combine:
- **Neural networks**: For flexible function approximation
- **Variational inference**: For tractable posterior approximation
- **Latent variable models**: For unsupervised learning

**Architecture**:
```
Input x → Encoder q(z|x) → Latent z → Decoder p(x|z) → Reconstructed x̂
```

**Training Objective (ELBO)**:
$$\mathcal{L} = \mathbb{E}_{q(z|x)}[\log p(x|z)] - D_{KL}(q(z|x) || p(z))$$

This framework enables:
- Generative modeling
- Dimensionality reduction
- Disentangled representation learning
- Anomaly detection

### Sub-part 2: dSprites Dataset

The **dSprites** dataset is a dataset of 2D shapes procedurally generated from 6 ground truth independent latent factors:
- **Shape**: 3 values (square, ellipse, heart)
- **Scale**: 6 values
- **Orientation**: 40 values  
- **Position X**: 32 values
- **Position Y**: 32 values
- **Color**: 1 value (white)

Total images: 737,280 images of size 64×64 pixels

This dataset is ideal for studying disentangled representations.

In [ ]:
import numpy as np

def load_dsprites(path='dsprites_ndarray_co1sh3sc6or40x32y32_64x64.npz'):
    try:
        data = np.load(path, allow_pickle=True, encoding='bytes')
        imgs = data['imgs']
        latents_values = data['latents_values']
        latents_classes = data['latents_classes']
        metadata = data['metadata'][()]
        
        print("dSprites Dataset Loaded Successfully!")
        print(f"Images shape: {imgs.shape}")
        print(f"Latents values shape: {latents_values.shape}")
        print(f"Latents classes shape: {latents_classes.shape}")
        print(f"\nLatent factors: {metadata[b'latents_names']}")
        print(f"Latent sizes: {metadata[b'latents_sizes']}")
        
        return imgs, latents_values, latents_classes, metadata
    except FileNotFoundError:
        print("Dataset not found. Please download from:")
        print("https://github.com/deepmind/dsprites-dataset/raw/master/dsprites_ndarray_co1sh3sc6or40x32y32_64x64.npz")
        return None, None, None, None

def visualize_dsprites_samples(imgs, n_samples=16):
    fig, axes = plt.subplots(4, 4, figsize=(10, 10))
    axes = axes.flatten()
    
    indices = np.random.choice(len(imgs), n_samples, replace=False)
    
    for idx, ax in zip(indices, axes):
        ax.imshow(imgs[idx], cmap='gray')
        ax.axis('off')
    
    plt.suptitle('dSprites Dataset - Random Samples', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('dsprites_samples.png', dpi=300, bbox_inches='tight')
    plt.show()

imgs, latents_values, latents_classes, metadata = load_dsprites()

if imgs is not None:
    visualize_dsprites_samples(imgs)


### Dataset Loading and Visualization

The **dSprites** dataset is ideal for studying disentanglement because:
- **Ground truth factors**: We know the exact generative factors
- **Independent factors**: Changes in one factor don't affect others
- **Combinatorial**: 737,280 images from systematic variation
- **Simple shapes**: Easy to visualize and interpret

This controlled setting allows us to measure how well our VAE learns to separate the underlying factors of variation.

### Sub-part 3: Reparameterization Trick

**Problem**: In VAE, we sample from $q(z|x)$ in the encoder. Direct sampling breaks the gradient flow, making backpropagation impossible.

**Solution - Reparameterization Trick**:

Instead of sampling $z \sim \mathcal{N}(\mu, \sigma^2)$ directly, we:
1. Sample $\epsilon \sim \mathcal{N}(0, 1)$
2. Compute $z = \mu + \sigma \odot \epsilon$

This way:
- The randomness is in $\epsilon$ (fixed, not learned)
- The gradient can flow through $\mu$ and $\sigma$
- The sampling operation becomes differentiable

**Mathematically**:
$$z = \mu(x) + \sigma(x) \odot \epsilon, \quad \epsilon \sim \mathcal{N}(0, I)$$

### Sub-part 4: VAE Implementation and Training

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class Encoder(nn.Module):
    def __init__(self, h_dim=256):
        super(Encoder, self).__init__()
        self.h_dim = h_dim
        
        self.conv1 = nn.Conv2d(1, 32, kernel_size=4, stride=2, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1)
        
        self.fc = nn.Linear(8192, h_dim * 2)
        
    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.relu(self.conv3(x))
        
        x = x.view(x.size(0), -1)
        
        h = self.fc(x)
        
        mu, log_var = torch.chunk(h, 2, dim=1)
        
        return mu, log_var


class Decoder(nn.Module):
    def __init__(self, h_dim=256):
        super(Decoder, self).__init__()
        self.h_dim = h_dim
        
        self.fc = nn.Linear(h_dim, 8192)
        
        self.deconv1 = nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1)
        self.deconv2 = nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1)
        self.deconv3 = nn.ConvTranspose2d(32, 1, kernel_size=4, stride=2, padding=1)
        
    def forward(self, z):
        x = self.fc(z)
        x = F.relu(x)
        
        x = x.view(x.size(0), 128, 8, 8)
        
        x = F.relu(self.deconv1(x))
        x = F.relu(self.deconv2(x))
        x = torch.sigmoid(self.deconv3(x))
        
        return x


class VAE(nn.Module):
    def __init__(self, h_dim=256):
        super(VAE, self).__init__()
        self.h_dim = h_dim
        
        self.encoder = Encoder(h_dim)
        self.decoder = Decoder(h_dim)
        
    def reparameterize(self, mu, log_var):
        std = torch.exp(0.5 * log_var)
        eps = torch.randn_like(std)
        z = mu + eps * std
        return z
    
    def forward(self, x):
        mu, log_var = self.encoder(x)
        
        z = self.reparameterize(mu, log_var)
        
        x_recon = self.decoder(z)
        
        return x_recon, mu, log_var
    
    def sample(self, num_samples, device):
        z = torch.randn(num_samples, self.h_dim).to(device)
        samples = self.decoder(z)
        return samples


model = VAE(h_dim=256).to(device)
print("VAE Model Architecture:")
print("="*60)
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")


### VAE Architecture Implementation

**Encoder Network** (Recognition model):
- Convolutional layers extract spatial features
- Outputs μ(x) and log σ²(x) for latent distribution
- Each input gets mapped to a distribution, not a point

**Decoder Network** (Generative model):
- Fully connected layer upsamples latent code
- Transposed convolutions reconstruct spatial structure
- Sigmoid output for binary images (pixel values in [0,1])

**Reparameterization Trick**:
- Sample ε ~ N(0,I)
- Compute z = μ + σ ⊙ ε
- Gradients flow through μ and σ, not through the sampling operation

This architecture balances:
- **Expressiveness**: Deep networks for complex patterns
- **Tractability**: Gaussian posteriors for closed-form KL
- **Efficiency**: Convolutional layers for spatial data

In [ ]:
from torch.utils.data import Dataset, DataLoader

class dSpritesDataset(Dataset):
    def __init__(self, imgs, transform=None):
        self.imgs = imgs
        self.transform = transform
        
    def __len__(self):
        return len(self.imgs)
    
    def __getitem__(self, idx):
        img = self.imgs[idx]
        img = torch.FloatTensor(img).unsqueeze(0)
        
        if self.transform:
            img = self.transform(img)
            
        return img


def create_dataloaders(imgs, batch_size=128, train_split=0.9):
    n_train = int(len(imgs) * train_split)
    indices = np.random.permutation(len(imgs))
    train_indices = indices[:n_train]
    val_indices = indices[n_train:]
    
    train_dataset = dSpritesDataset(imgs[train_indices])
    val_dataset = dSpritesDataset(imgs[val_indices])
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, 
                             shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, 
                           shuffle=False, num_workers=2, pin_memory=True)
    
    print(f"Training samples: {len(train_dataset)}")
    print(f"Validation samples: {len(val_dataset)}")
    
    return train_loader, val_loader


if imgs is not None:
    subset_size = 50000
    imgs_subset = imgs[np.random.choice(len(imgs), subset_size, replace=False)]
    train_loader, val_loader = create_dataloaders(imgs_subset, batch_size=128)


### Dataset Preparation

Custom PyTorch dataset and dataloaders for efficient training:

**Features**:
- **Memory efficiency**: Load data on-demand, not all at once
- **Batching**: Process multiple images in parallel on GPU
- **Shuffling**: Random order each epoch to avoid overfitting patterns
- **Train/validation split**: Monitor generalization performance

**Best Practices**:
- Use `pin_memory=True` for faster GPU transfer
- Set `num_workers > 0` for parallel data loading
- Normalize images to [0,1] for stable training

In [ ]:
from tqdm import tqdm

def vae_loss(x_recon, x, mu, log_var, beta=1.0):
    recon_loss = F.binary_cross_entropy(x_recon, x, reduction='sum')
    
    kl_loss = -0.5 * torch.sum(1 + log_var - mu.pow(2) - log_var.exp())
    
    total_loss = recon_loss + beta * kl_loss
    
    return total_loss, recon_loss, kl_loss


def train_epoch(model, train_loader, optimizer, beta=1.0):
    model.train()
    total_loss = 0
    total_recon = 0
    total_kl = 0
    
    pbar = tqdm(train_loader, desc='Training')
    for batch_idx, data in enumerate(pbar):
        data = data.to(device)
        
        optimizer.zero_grad()
        
        x_recon, mu, log_var = model(data)
        
        loss, recon_loss, kl_loss = vae_loss(x_recon, data, mu, log_var, beta)
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        total_recon += recon_loss.item()
        total_kl += kl_loss.item()
        
        pbar.set_postfix({
            'loss': loss.item() / len(data),
            'recon': recon_loss.item() / len(data),
            'kl': kl_loss.item() / len(data)
        })
    
    n_samples = len(train_loader.dataset)
    return total_loss / n_samples, total_recon / n_samples, total_kl / n_samples


def validate(model, val_loader, beta=1.0):
    model.eval()
    total_loss = 0
    total_recon = 0
    total_kl = 0
    
    with torch.no_grad():
        for data in val_loader:
            data = data.to(device)
            
            x_recon, mu, log_var = model(data)
            
            loss, recon_loss, kl_loss = vae_loss(x_recon, data, mu, log_var, beta)
            
            total_loss += loss.item()
            total_recon += recon_loss.item()
            total_kl += kl_loss.item()
    
    n_samples = len(val_loader.dataset)
    return total_loss / n_samples, total_recon / n_samples, total_kl / n_samples


print("Loss functions and training utilities defined!")


### Loss Functions

**VAE Loss = Reconstruction Loss + β × KL Loss**

**1. Reconstruction Loss** (Negative log-likelihood):
- Binary cross-entropy for binary images
- Measures how well decoder reconstructs input
- Lower = better reconstruction quality

**2. KL Divergence Loss** (Regularization):
$$D_{KL}(q(z|x) || p(z)) = -\frac{1}{2} \sum_{j=1}^{d} (1 + \log \sigma_j^2 - \mu_j^2 - \sigma_j^2)$$
- Closed-form for Gaussian distributions
- Encourages latent distribution to match prior N(0,I)
- Lower = latent space more aligned with prior

**β Parameter**:
- Controls trade-off between reconstruction and regularization
- β > 1: More disentanglement, worse reconstruction
- β < 1: Better reconstruction, less disentanglement
- β = 1: Standard VAE (ELBO)

In [ ]:
def train_vae(model, train_loader, val_loader, epochs=50, lr=0.001, beta=1.0, save_path='vae_model.pth'):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=5, verbose=True
    )
    
    history = {
        'train_loss': [], 'train_recon': [], 'train_kl': [],
        'val_loss': [], 'val_recon': [], 'val_kl': []
    }
    
    best_val_loss = float('inf')
    
    print(f"\nTraining VAE (β={beta})")
    print("="*80)
    
    for epoch in range(epochs):
        print(f"\nEpoch {epoch+1}/{epochs}")
        
        train_loss, train_recon, train_kl = train_epoch(model, train_loader, optimizer, beta)
        
        val_loss, val_recon, val_kl = validate(model, val_loader, beta)
        
        scheduler.step(val_loss)
        
        history['train_loss'].append(train_loss)
        history['train_recon'].append(train_recon)
        history['train_kl'].append(train_kl)
        history['val_loss'].append(val_loss)
        history['val_recon'].append(val_recon)
        history['val_kl'].append(val_kl)
        
        print(f"\nEpoch {epoch+1} Summary:")
        print(f"  Train - Loss: {train_loss:.4f}, Recon: {train_recon:.4f}, KL: {train_kl:.4f}")
        print(f"  Val   - Loss: {val_loss:.4f}, Recon: {val_recon:.4f}, KL: {val_kl:.4f}")
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_loss': val_loss,
                'beta': beta
            }, save_path)
            print(f"  ✓ Best model saved (val_loss: {val_loss:.4f})")
    
    print("\n" + "="*80)
    print("Training completed!")
    
    return history


print("Training function ready. Uncomment to start training.")


### Training Loop

Complete training pipeline with:

**Optimization**:
- Adam optimizer with adaptive learning rates
- Learning rate scheduling (ReduceLROnPlateau)
- Gradient descent on negative ELBO

**Training Strategy**:
1. Forward pass through encoder and decoder
2. Compute reconstruction + KL loss
3. Backpropagate gradients
4. Update weights
5. Validate on held-out data

**Monitoring**:
- Track total loss, reconstruction loss, and KL divergence separately
- Save best model based on validation loss
- Use tqdm for progress visualization

**Early Stopping Criteria**:
- Learning rate plateaus
- Validation loss stops improving
- Maximum epochs reached

In [ ]:
from sklearn.decomposition import PCA

def plot_training_history(history, beta, save_path=None):
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    epochs = range(1, len(history['train_loss']) + 1)
    
    axes[0].plot(epochs, history['train_loss'], 'b-', label='Train', linewidth=2)
    axes[0].plot(epochs, history['val_loss'], 'r-', label='Validation', linewidth=2)
    axes[0].set_xlabel('Epoch', fontsize=12)
    axes[0].set_ylabel('Total Loss', fontsize=12)
    axes[0].set_title('Total Loss', fontsize=14, fontweight='bold')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    axes[1].plot(epochs, history['train_recon'], 'b-', label='Train', linewidth=2)
    axes[1].plot(epochs, history['val_recon'], 'r-', label='Validation', linewidth=2)
    axes[1].set_xlabel('Epoch', fontsize=12)
    axes[1].set_ylabel('Reconstruction Loss', fontsize=12)
    axes[1].set_title('Reconstruction Loss', fontsize=14, fontweight='bold')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    axes[2].plot(epochs, history['train_kl'], 'b-', label='Train', linewidth=2)
    axes[2].plot(epochs, history['val_kl'], 'r-', label='Validation', linewidth=2)
    axes[2].set_xlabel('Epoch', fontsize=12)
    axes[2].set_ylabel('KL Divergence', fontsize=12)
    axes[2].set_title(f'KL Divergence (β={beta})', fontsize=14, fontweight='bold')
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)
    
    plt.suptitle(f'Training History (β={beta})', fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()


def visualize_reconstructions(model, data_loader, n_samples=8, save_path=None):
    model.eval()
    
    data = next(iter(data_loader))
    data = data[:n_samples].to(device)
    
    with torch.no_grad():
        recon, _, _ = model(data)
    
    data = data.cpu().numpy()
    recon = recon.cpu().numpy()
    
    fig, axes = plt.subplots(2, n_samples, figsize=(n_samples*2, 4))
    
    for i in range(n_samples):
        axes[0, i].imshow(data[i, 0], cmap='gray')
        axes[0, i].axis('off')
        if i == 0:
            axes[0, i].set_ylabel('Original', fontsize=12, fontweight='bold')
        
        axes[1, i].imshow(recon[i, 0], cmap='gray')
        axes[1, i].axis('off')
        if i == 0:
            axes[1, i].set_ylabel('Reconstructed', fontsize=12, fontweight='bold')
    
    plt.suptitle('Original vs Reconstructed Images', fontsize=14, fontweight='bold')
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()


def visualize_latent_space_2d(model, data_loader, latents_classes=None, save_path=None):
    model.eval()
    
    latent_vectors = []
    labels = []
    
    with torch.no_grad():
        for idx, data in enumerate(data_loader):
            if idx >= 50:
                break
            data = data.to(device)
            mu, _ = model.encoder(data)
            latent_vectors.append(mu.cpu().numpy())
    
    latent_vectors = np.concatenate(latent_vectors, axis=0)
    
    pca = PCA(n_components=2)
    latent_2d = pca.fit_transform(latent_vectors)
    
    plt.figure(figsize=(10, 8))
    plt.scatter(latent_2d[:, 0], latent_2d[:, 1], alpha=0.5, s=10)
    plt.xlabel('PC1', fontsize=12)
    plt.ylabel('PC2', fontsize=12)
    plt.title('Latent Space Visualization (PCA)', fontsize=14, fontweight='bold')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"Explained variance ratio: {pca.explained_variance_ratio_}")
    
    return pca, latent_2d


print("Visualization functions defined!")


### Visualization Tools

**Training History Plot**:
- Monitor convergence by plotting loss curves
- Identify overfitting (train/val gap)
- See effect of β on KL divergence

**Reconstruction Visualization**:
- Compare original vs reconstructed images
- Assess perceptual quality
- Check for mode collapse or artifacts

**Latent Space Visualization**:
- Use PCA to reduce high-dimensional latent space to 2D
- Color by ground truth factors to assess disentanglement
- Look for structure and clustering

**Best Practices**:
- Save all visualizations for later comparison
- Use consistent color schemes and layouts
- Generate visualizations at regular intervals during training

### Sub-part 5: β-VAE

**β-VAE** modifies the VAE loss by adding a weight β to the KL term:

$$\mathcal{L}_{\beta-VAE} = \mathbb{E}_{q(z|x)}[\log p(x|z)] - \beta \cdot D_{KL}(q(z|x) || p(z))$$

**Improvements over standard VAE:**
1. **Better disentanglement**: Higher β encourages independence between latent dimensions
2. **Controlled trade-off**: β balances reconstruction quality vs disentanglement
3. **More structured latent space**: Forces the model to use latent dimensions more efficiently

- **β < 1**: Emphasizes reconstruction (better quality, less disentanglement)
- **β = 1**: Standard VAE
- **β > 1**: Emphasizes disentanglement (worse reconstruction, better interpretability)

---

## β-VAE: Improving Disentanglement

### Theoretical Foundation

**Standard VAE**: Maximizes ELBO with β = 1
$$\mathcal{L} = \mathbb{E}_{q(z|x)}[\log p(x|z)] - D_{KL}(q(z|x) || p(z))$$

**β-VAE**: Adds weight to KL term
$$\mathcal{L}_{\beta} = \mathbb{E}_{q(z|x)}[\log p(x|z)] - \beta \cdot D_{KL}(q(z|x) || p(z))$$

### Why β > 1 Helps Disentanglement

**Information Bottleneck**: Higher β forces the model to:
1. Use latent capacity more efficiently
2. Encode only the most important factors
3. Prevent redundant encoding across dimensions

**Independence Pressure**: Strong KL penalty encourages:
- Latent dimensions to be independent (diagonal covariance)
- Each dimension to capture a single factor of variation
- Sparse activation patterns

**Trade-off**:
- **β = 1**: Optimal for reconstruction, poor disentanglement
- **β = 2-4**: Balanced trade-off
- **β = 5-10**: Strong disentanglement, degraded reconstruction
- **β > 10**: Over-regularized, loses important information

### Sub-part 6: Train β-VAE with Different β Values

Let's train models with different β values and compare results.

In [ ]:
# Train β-VAE with different β values

# Example training code (uncomment to run)
"""
# Train with β = 2 (lower, better reconstruction)
model_beta2 = VAE(h_dim=256).to(device)
history_beta2 = train_vae(model_beta2, train_loader, val_loader, 
                          epochs=50, lr=0.001, beta=2.0, 
                          save_path='vae_beta2.pth')
plot_training_history(history_beta2, beta=2.0, save_path='training_history_beta2.png')
visualize_reconstructions(model_beta2, val_loader, save_path='reconstructions_beta2.png')

# Train with β = 5 (higher, better disentanglement)
model_beta5 = VAE(h_dim=256).to(device)
history_beta5 = train_vae(model_beta5, train_loader, val_loader, 
                          epochs=50, lr=0.001, beta=5.0, 
                          save_path='vae_beta5.pth')
plot_training_history(history_beta5, beta=5.0, save_path='training_history_beta5.png')
visualize_reconstructions(model_beta5, val_loader, save_path='reconstructions_beta5.png')
"""

print("β-VAE training code ready.")
print("Suggested β values: β=2 (lower) and β=5 (higher)")
print("Uncomment the code above to train models with different β values.")

### Experimental Setup for β-VAE

To properly evaluate the effect of β, we should train multiple models:

**Recommended β values**:
- **β = 1**: Baseline (standard VAE)
- **β = 2**: Mild regularization
- **β = 5**: Strong regularization
- **β = 10**: Very strong (optional)

**What to compare**:
1. **Reconstruction quality**: Visual inspection + MSE
2. **Disentanglement**: MIG metric (next section)
3. **Latent space structure**: PCA visualization
4. **Training dynamics**: Loss curves

**Expected Observations**:
- Higher β → higher reconstruction loss
- Higher β → lower KL divergence (closer to prior)
- Higher β → better MIG scores
- Higher β → more interpretable latent traversals

### Sub-part 7: MIG (Mutual Information Gap) Metric

**MIG** measures how well the latent dimensions are disentangled by computing:

$$MIG = \frac{1}{K} \sum_{k=1}^{K} \frac{I(z_j; v_k) - I(z_{j'}; v_k)}{H(v_k)}$$

where:
- $v_k$ are the ground truth factors
- $z_j$ is the latent dimension with highest mutual information with $v_k$
- $z_{j'}$ is the dimension with second-highest MI
- Higher MIG (closer to 1) = better disentanglement

In [ ]:
from sklearn.metrics import mutual_info_score

def discretize(data, num_bins=20):
    data_min = data.min()
    data_max = data.max()
    bins = np.linspace(data_min, data_max, num_bins + 1)
    discretized = np.digitize(data, bins) - 1
    discretized = np.clip(discretized, 0, num_bins - 1)
    return discretized


def compute_mutual_information_matrix(latents, factors):
    n_latent = latents.shape[1]
    n_factors = factors.shape[1]
    
    mi_matrix = np.zeros((n_latent, n_factors))
    
    latents_discrete = np.zeros_like(latents, dtype=int)
    for i in range(n_latent):
        latents_discrete[:, i] = discretize(latents[:, i])
    
    for i in range(n_latent):
        for j in range(n_factors):
            mi_matrix[i, j] = mutual_info_score(latents_discrete[:, i], factors[:, j])
    
    return mi_matrix


def compute_mig(latents, factors):
    mi_matrix = compute_mutual_information_matrix(latents, factors)
    
    n_factors = factors.shape[1]
    mig_per_factor = np.zeros(n_factors)
    
    factor_entropy = np.zeros(n_factors)
    for j in range(n_factors):
        _, counts = np.unique(factors[:, j], return_counts=True)
        probs = counts / counts.sum()
        factor_entropy[j] = -np.sum(probs * np.log(probs + 1e-10))
    
    for j in range(n_factors):
        mi_sorted = np.sort(mi_matrix[:, j])[::-1]
        
        if len(mi_sorted) >= 2:
            gap = mi_sorted[0] - mi_sorted[1]
        else:
            gap = mi_sorted[0]
        
        if factor_entropy[j] > 0:
            mig_per_factor[j] = gap / factor_entropy[j]
        else:
            mig_per_factor[j] = 0
    
    mig_score = np.mean(mig_per_factor)
    
    return mig_score, mig_per_factor, mi_matrix


def extract_latents(model, data_loader, max_samples=10000):
    model.eval()
    latents = []
    
    with torch.no_grad():
        for data in data_loader:
            if len(latents) * data.size(0) >= max_samples:
                break
            data = data.to(device)
            mu, _ = model.encoder(data)
            latents.append(mu.cpu().numpy())
    
    latents = np.concatenate(latents, axis=0)[:max_samples]
    return latents


def evaluate_disentanglement(model, imgs, latents_classes, max_samples=10000):
    print("\nEvaluating Disentanglement (MIG Metric)")
    print("="*60)
    
    subset_indices = np.random.choice(len(imgs), min(max_samples, len(imgs)), replace=False)
    temp_dataset = dSpritesDataset(imgs[subset_indices])
    temp_loader = DataLoader(temp_dataset, batch_size=128, shuffle=False)
    
    print("Extracting latent representations...")
    latents = extract_latents(model, temp_loader, max_samples)
    
    factors = latents_classes[subset_indices]
    
    print("Computing MIG metric...")
    mig_score, mig_per_factor, mi_matrix = compute_mig(latents, factors)
    
    print(f"\nOverall MIG Score: {mig_score:.4f}")
    print("\nMIG per factor:")
    factor_names = ['color', 'shape', 'scale', 'orientation', 'posX', 'posY']
    for i, (name, score) in enumerate(zip(factor_names, mig_per_factor)):
        print(f"  {name:12s}: {score:.4f}")
    
    plt.figure(figsize=(10, 6))
    plt.imshow(mi_matrix, aspect='auto', cmap='viridis')
    plt.colorbar(label='Mutual Information')
    plt.xlabel('Ground Truth Factors', fontsize=12)
    plt.ylabel('Latent Dimensions', fontsize=12)
    plt.title('Mutual Information Matrix', fontsize=14, fontweight='bold')
    plt.xticks(range(len(factor_names)), factor_names, rotation=45)
    plt.tight_layout()
    plt.savefig('mi_matrix.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    return mig_score, mig_per_factor, mi_matrix


print("MIG metric implementation ready!")


### MIG (Mutual Information Gap) Metric

MIG quantitatively measures disentanglement quality.

**Algorithm**:
1. **Discretize** continuous latent codes into bins
2. **Compute MI** between each latent dimension and each ground truth factor
3. **Find gap**: For each factor, find difference between highest and second-highest MI
4. **Normalize** by factor entropy
5. **Average** across all factors

**Mathematical Definition**:
$$MIG = \frac{1}{K} \sum_{k=1}^{K} \frac{1}{H(v_k)} (I(z_{j_1}; v_k) - I(z_{j_2}; v_k))$$

where:
- $v_k$: k-th ground truth factor
- $z_{j_1}$: latent dimension with highest MI with $v_k$
- $z_{j_2}$: latent dimension with second-highest MI
- $H(v_k)$: Entropy of factor $v_k$

**Interpretation**:
- **MIG = 0**: No disentanglement (random)
- **MIG = 0.1-0.3**: Weak disentanglement
- **MIG = 0.3-0.6**: Moderate disentanglement
- **MIG > 0.6**: Strong disentanglement
- **MIG = 1**: Perfect disentanglement (each factor mapped to unique latent dim)

**Why the "Gap"?**
The gap measures how much more information the best latent dimension provides compared to the second-best. A large gap indicates that one dimension clearly captures that factor, not spread across multiple dimensions.

## Part 2: Theoretical Questions on VAE Variants

### Sub-part 1: VQ-VAE (Vector Quantized VAE)

**VQ-VAE** improves the latent space by introducing discrete latent variables through vector quantization.

**Key Differences from Standard VAE:**

1. **Discrete Latent Space**: Instead of continuous latent vectors, VQ-VAE uses discrete codes from a learned codebook
2. **Codebook**: Contains a finite set of embedding vectors $\{e_k\}_{k=1}^K$
3. **Quantization**: Encoder output is mapped to nearest codebook vector

**Process:**
$$z_e = \text{Encoder}(x)$$
$$z_q = e_k, \quad k = \arg\min_j ||z_e - e_j||_2$$
$$\hat{x} = \text{Decoder}(z_q)$$

**Advantages:**
- More structured latent space
- No "posterior collapse" problem
- Better for autoregressive modeling (e.g., with PixelCNN)
- Can be used as discrete tokens for language-like modeling

**Discretization Meaning:**
- Latent space becomes a discrete set of learned representations
- Each input maps to one of K discrete codes
- Enables discrete reasoning and combinatorial generalization

---

## Advanced VAE Variants

This section covers three important VAE variants that address different limitations of the standard VAE:

1. **VQ-VAE**: Discrete latent space for better structure
2. **VampPrior**: Learnable, data-dependent prior
3. **SC-VAE**: Sparse coding for interpretability

Each variant introduces novel ideas that have influenced modern generative models (e.g., DALL-E uses VQ-VAE principles).

### Sub-part 2: VampPrior

**VampPrior** (Variational Mixture of Posteriors Prior) proposes a more flexible prior for VAE.

**Standard VAE Prior:** $p(z) = \mathcal{N}(0, I)$

**VampPrior:** 
$$p(z) = \frac{1}{K} \sum_{k=1}^{K} q(z|u_k)$$

where $\{u_k\}_{k=1}^K$ are learnable pseudo-inputs.

**How They Estimate Posterior:**
1. **Pseudo-inputs**: Learn K pseudo-inputs $\{u_k\}$ from data
2. **Mixture of posteriors**: Prior is mixture of encoder posteriors evaluated at pseudo-inputs
3. **Training**: Pseudo-inputs are learned during training

**Advantages:**
- **More expressive prior**: Can capture multimodal distributions
- **Better latent space utilization**: Avoids "holes" in latent space
- **Data-adaptive**: Prior adapts to the actual data distribution
- **Reduced KL**: Better match between prior and aggregate posterior $q(z) = \mathbb{E}_{p(x)}[q(z|x)]$

**Why Better Than Standard Normal:**
- Standard $\mathcal{N}(0, I)$ may not match the true aggregate posterior
- VampPrior is more flexible and data-dependent
- Leads to better use of latent variables

### VampPrior in Detail

**Problem with Standard Prior**:
The standard prior p(z) = N(0, I) is fixed and may not match the aggregate posterior:
$$q(z) = \mathbb{E}_{p_{\text{data}}(x)}[q(z|x)]$$

This mismatch leads to:
- Unused latent dimensions ("holes" in latent space)
- Suboptimal ELBO
- Poor generalization

**VampPrior Solution**:
Learn K pseudo-inputs {u₁, ..., uₖ} and define:
$$p(z) = \frac{1}{K} \sum_{k=1}^{K} q(z|u_k)$$

**Benefits**:
1. **Multimodal**: Can represent complex distributions
2. **Adaptive**: Learns from data, not fixed
3. **No posterior collapse**: Better latent space utilization
4. **Tighter ELBO**: Closer match to aggregate posterior

**Implementation**:
- Initialize pseudo-inputs as learnable parameters
- During training, update both model parameters and pseudo-inputs
- Prior becomes mixture of Gaussians centered at encoder outputs for pseudo-inputs

### Sub-part 3: SC-VAE (Sparse Coding VAE)

**SC-VAE** introduces sparse coding into VAE to encourage sparse latent representations.

**Key Components:**

1. **ISTA (Iterative Shrinkage-Thresholding Algorithm)**:
   - Optimization algorithm for sparse coding
   - Applies soft thresholding to encourage sparsity
   - Iteratively refines latent codes

2. **Sparse Representation**: 
   $$\min_z \frac{1}{2}||x - Dz||_2^2 + \lambda||z||_1$$
   
   where D is dictionary (decoder) and λ controls sparsity

**Role of ISTA:**
- Learns to encode data into sparse representations
- Unrolled ISTA layers replace standard encoder
- Differentiable, so can be trained end-to-end

**Advantages of Sparsity:**

1. **Interpretability**: Only few latent dimensions active for each input
2. **Efficiency**: Compact representations
3. **Disentanglement**: Sparse codes often align with semantic factors
4. **Robustness**: Less sensitive to noise
5. **Biological plausibility**: Brain uses sparse coding

**Comparison to Standard VAE:**
- Standard VAE: Dense latent representations
- SC-VAE: Sparse, structured latent codes
- Better for interpretable and disentangled representations

### SC-VAE and Sparse Coding

**Sparse Coding Principle**:
Natural signals can be represented as sparse linear combinations of basis functions:
$$x ≈ Dz, \quad \text{where } z \text{ is sparse}$$

**ISTA (Iterative Shrinkage-Thresholding Algorithm)**:
Solves the L1-regularized least squares problem:
$$\min_z \frac{1}{2}||x - Dz||_2^2 + \lambda||z||_1$$

**ISTA Update Rule**:
$$z^{(t+1)} = \text{SoftThreshold}(z^{(t)} + D^T(x - Dz^{(t)}), \lambda)$$

where SoftThreshold(x, λ) = sign(x) × max(|x| - λ, 0)

**SC-VAE Architecture**:
- **Encoder**: Unrolled ISTA layers (T iterations)
- **Decoder**: Standard neural network (dictionary D)
- **Differentiable**: End-to-end training via backprop through ISTA

**Why Sparsity Helps**:
1. **Interpretability**: Only few dimensions active per sample
2. **Disentanglement**: Each basis often captures semantic factors
3. **Efficiency**: Compact representations
4. **Robustness**: Less sensitive to noise
5. **Neuroscience**: Consistent with brain's sparse coding

**Comparison**:
- Standard VAE: Dense latent codes (many dimensions active)
- SC-VAE: Sparse latent codes (few dimensions active)
- Better for: Interpretability, feature selection, anomaly detection

## Example Training Pipeline

Below is a complete example of how to train and evaluate the models:

In [ ]:
# Complete Training and Evaluation Pipeline
# Uncomment sections as needed

"""
# ============================================================================
# STEP 1: Load Dataset
# ============================================================================
imgs, latents_values, latents_classes, metadata = load_dsprites()

if imgs is not None:
    # Visualize samples
    visualize_dsprites_samples(imgs, n_samples=16)
    
    # Create dataloaders
    train_loader, val_loader = create_dataloaders(imgs, batch_size=128)
    
    # ========================================================================
    # STEP 2: Train Standard VAE (β=1)
    # ========================================================================
    print("\n" + "="*80)
    print("Training Standard VAE (β=1)")
    print("="*80)
    
    model_beta1 = VAE(h_dim=256).to(device)
    history_beta1 = train_vae(
        model_beta1, train_loader, val_loader,
        epochs=50, lr=0.001, beta=1.0,
        save_path='vae_beta1.pth'
    )
    
    # Plot training history
    plot_training_history(history_beta1, beta=1.0, 
                         save_path='history_beta1.png')
    
    # Visualize reconstructions
    visualize_reconstructions(model_beta1, val_loader, n_samples=8,
                            save_path='recon_beta1.png')
    
    # ========================================================================
    # STEP 3: Train β-VAE with β=2 (Lower)
    # ========================================================================
    print("\n" + "="*80)
    print("Training β-VAE (β=2)")
    print("="*80)
    
    model_beta2 = VAE(h_dim=256).to(device)
    history_beta2 = train_vae(
        model_beta2, train_loader, val_loader,
        epochs=50, lr=0.001, beta=2.0,
        save_path='vae_beta2.pth'
    )
    
    plot_training_history(history_beta2, beta=2.0, 
                         save_path='history_beta2.png')
    visualize_reconstructions(model_beta2, val_loader, n_samples=8,
                            save_path='recon_beta2.png')
    
    # ========================================================================
    # STEP 4: Train β-VAE with β=5 (Higher)
    # ========================================================================
    print("\n" + "="*80)
    print("Training β-VAE (β=5)")
    print("="*80)
    
    model_beta5 = VAE(h_dim=256).to(device)
    history_beta5 = train_vae(
        model_beta5, train_loader, val_loader,
        epochs=50, lr=0.001, beta=5.0,
        save_path='vae_beta5.pth'
    )
    
    plot_training_history(history_beta5, beta=5.0, 
                         save_path='history_beta5.png')
    visualize_reconstructions(model_beta5, val_loader, n_samples=8,
                            save_path='recon_beta5.png')
    
    # ========================================================================
    # STEP 5: Evaluate Disentanglement with MIG
    # ========================================================================
    print("\n" + "="*80)
    print("Evaluating Disentanglement (MIG Metric)")
    print("="*80)
    
    # Evaluate all models
    models = {
        'VAE (β=1)': model_beta1,
        'β-VAE (β=2)': model_beta2,
        'β-VAE (β=5)': model_beta5
    }
    
    mig_results = {}
    for name, model in models.items():
        print(f"\n{name}:")
        mig_score, mig_per_factor, mi_matrix = evaluate_disentanglement(
            model, imgs, latents_classes, max_samples=10000
        )
        mig_results[name] = {
            'mig_score': mig_score,
            'mig_per_factor': mig_per_factor
        }
    
    # Compare MIG scores
    print("\n" + "="*80)
    print("MIG Score Comparison")
    print("="*80)
    for name, results in mig_results.items():
        print(f"{name:20s}: {results['mig_score']:.4f}")
    
    # ========================================================================
    # STEP 6: PCA Visualization
    # ========================================================================
    print("\n" + "="*80)
    print("Latent Space Visualization (PCA)")
    print("="*80)
    
    for name, model in models.items():
        print(f"\n{name}:")
        pca, latent_2d = visualize_latent_space_2d(
            model, val_loader, latents_classes,
            save_path=f'pca_{name.replace(" ", "_")}.png'
        )
    
    # ========================================================================
    # STEP 7: Compare Reconstructions Side-by-Side
    # ========================================================================
    print("\n" + "="*80)
    print("Comparing Reconstructions")
    print("="*80)
    
    # Get sample batch
    sample_data = next(iter(val_loader))[:8].to(device)
    
    fig, axes = plt.subplots(4, 8, figsize=(16, 8))
    
    with torch.no_grad():
        recon1, _, _ = model_beta1(sample_data)
        recon2, _, _ = model_beta2(sample_data)
        recon5, _, _ = model_beta5(sample_data)
    
    for i in range(8):
        # Original
        axes[0, i].imshow(sample_data[i, 0].cpu(), cmap='gray')
        axes[0, i].axis('off')
        if i == 0:
            axes[0, i].set_ylabel('Original', fontsize=10)
        
        # β=1
        axes[1, i].imshow(recon1[i, 0].cpu(), cmap='gray')
        axes[1, i].axis('off')
        if i == 0:
            axes[1, i].set_ylabel('β=1', fontsize=10)
        
        # β=2
        axes[2, i].imshow(recon2[i, 0].cpu(), cmap='gray')
        axes[2, i].axis('off')
        if i == 0:
            axes[2, i].set_ylabel('β=2', fontsize=10)
        
        # β=5
        axes[3, i].imshow(recon5[i, 0].cpu(), cmap='gray')
        axes[3, i].axis('off')
        if i == 0:
            axes[3, i].set_ylabel('β=5', fontsize=10)
    
    plt.suptitle('Reconstruction Comparison: Different β Values', 
                fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('reconstruction_comparison.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("\n" + "="*80)
    print("Training and Evaluation Complete!")
    print("="*80)
"""

print("Complete training pipeline ready!")
print("Uncomment the code above to run the full experiment.")

---

## Complete Training and Evaluation Pipeline

This comprehensive pipeline executes all required tasks:

### Pipeline Steps

**1. Data Preparation**:
- Load dSprites dataset (737,280 images)
- Create train/validation split (90/10)
- Set up dataloaders with batching

**2. Model Training**:
- Train standard VAE (β=1)
- Train β-VAE with β=2 (mild regularization)
- Train β-VAE with β=5 (strong regularization)

**3. Visualization**:
- Plot training curves (total loss, reconstruction, KL)
- Show reconstruction quality for each model
- Generate latent space PCA plots

**4. Quantitative Evaluation**:
- Compute MIG scores for all models
- Generate mutual information matrices
- Compare disentanglement quality

**5. Analysis**:
- Side-by-side reconstruction comparison
- MIG score comparison across β values
- Identify which factors are best disentangled

### Expected Runtime

On GPU (CUDA):
- Dataset loading: ~30 seconds
- Training per model (50 epochs): ~20-30 minutes
- Evaluation (MIG): ~2-3 minutes per model
- Total: ~1.5-2 hours for complete pipeline

On CPU:
- Training per model: ~2-3 hours
- Total: ~6-8 hours

### Key Hyperparameters

**Architecture**:
- Latent dimension: 256
- Encoder: 3 conv layers (32→64→128 channels)
- Decoder: 3 transposed conv layers (128→64→32→1)

**Training**:
- Batch size: 128
- Learning rate: 0.001 (with ReduceLROnPlateau)
- Epochs: 50
- Optimizer: Adam

**β values**: [1, 2, 5]

---

## READY-TO-RUN TRAINING CODE

The following cells contain ready-to-execute code for all required experiments.
Simply run them sequentially to complete the assignment.

In [ ]:
# ============================================================================
# EXECUTABLE TRAINING CODE - Run this cell to train all models
# ============================================================================

import os
import warnings
warnings.filterwarnings('ignore')

# Configuration
TRAIN_MODELS = True  # Set to True to train models
EPOCHS = 50
BATCH_SIZE = 128
LEARNING_RATE = 0.001
LATENT_DIM = 256
DATA_SUBSET_SIZE = 50000  # Use subset for faster training, or None for full dataset

# Beta values to test
BETA_VALUES = [1.0, 2.0, 5.0]

print("="*80)
print("COMPREHENSIVE VAE TRAINING PIPELINE")
print("="*80)
print(f"Configuration:")
print(f"  Epochs: {EPOCHS}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Latent dimension: {LATENT_DIM}")
print(f"  Beta values: {BETA_VALUES}")
print(f"  Device: {device}")
print("="*80)

if TRAIN_MODELS:
    # ========================================================================
    # STEP 1: Load and prepare data
    # ========================================================================
    print("\n[STEP 1/6] Loading dSprites dataset...")
    
    try:
        # Try to load dataset
        data = np.load('dsprites_ndarray_co1sh3sc6or40x32y32_64x64.npz', 
                      allow_pickle=True, encoding='bytes')
        imgs = data['imgs']
        latents_values = data['latents_values']
        latents_classes = data['latents_classes']
        metadata = data['metadata'][()]
        
        print(f"✓ Dataset loaded successfully!")
        print(f"  Total images: {len(imgs):,}")
        print(f"  Image shape: {imgs.shape[1:]}")
        
        # Use subset if specified
        if DATA_SUBSET_SIZE and DATA_SUBSET_SIZE < len(imgs):
            indices = np.random.choice(len(imgs), DATA_SUBSET_SIZE, replace=False)
            imgs_train = imgs[indices]
            latents_classes_train = latents_classes[indices]
            print(f"  Using subset: {DATA_SUBSET_SIZE:,} images")
        else:
            imgs_train = imgs
            latents_classes_train = latents_classes
            print(f"  Using full dataset: {len(imgs_train):,} images")
        
        # Create dataloaders
        train_loader, val_loader = create_dataloaders(
            imgs_train, batch_size=BATCH_SIZE, train_split=0.9
        )
        
        print(f"✓ Dataloaders created")
        
    except FileNotFoundError:
        print("\n❌ ERROR: dSprites dataset not found!")
        print("\nPlease download it using:")
        print("wget https://github.com/deepmind/dsprites-dataset/raw/master/dsprites_ndarray_co1sh3sc6or40x32y32_64x64.npz")
        print("\nOr in Python:")
        print("import urllib.request")
        print("url = 'https://github.com/deepmind/dsprites-dataset/raw/master/dsprites_ndarray_co1sh3sc6or40x32y32_64x64.npz'")
        print("urllib.request.urlretrieve(url, 'dsprites_ndarray_co1sh3sc6or40x32y32_64x64.npz')")
        TRAIN_MODELS = False
    
    if TRAIN_MODELS:
        # Visualize samples
        visualize_dsprites_samples(imgs_train, n_samples=16)
        
        # ====================================================================
        # STEP 2-4: Train models with different beta values
        # ====================================================================
        trained_models = {}
        training_histories = {}
        
        for beta in BETA_VALUES:
            print(f"\n[STEP {BETA_VALUES.index(beta)+2}/6] Training VAE with β={beta}")
            print("="*80)
            
            # Create model
            model = VAE(h_dim=LATENT_DIM).to(device)
            
            # Train
            history = train_vae(
                model, train_loader, val_loader,
                epochs=EPOCHS,
                lr=LEARNING_RATE,
                beta=beta,
                save_path=f'vae_beta{beta}.pth'
            )
            
            # Store results
            trained_models[f'beta_{beta}'] = model
            training_histories[f'beta_{beta}'] = history
            
            # Plot training history
            plot_training_history(
                history, 
                beta=beta, 
                save_path=f'training_history_beta{beta}.png'
            )
            
            # Visualize reconstructions
            visualize_reconstructions(
                model, 
                val_loader, 
                n_samples=8,
                save_path=f'reconstructions_beta{beta}.png'
            )
            
            print(f"\n✓ Model with β={beta} training completed!")
            print(f"  Final train loss: {history['train_loss'][-1]:.4f}")
            print(f"  Final val loss: {history['val_loss'][-1]:.4f}")
            print(f"  Model saved: vae_beta{beta}.pth")
        
        # ====================================================================
        # STEP 5: Evaluate disentanglement with MIG
        # ====================================================================
        print(f"\n[STEP 5/6] Evaluating Disentanglement (MIG Metric)")
        print("="*80)
        
        mig_results = {}
        for beta in BETA_VALUES:
            model_key = f'beta_{beta}'
            print(f"\nEvaluating β={beta}...")
            
            mig_score, mig_per_factor, mi_matrix = evaluate_disentanglement(
                trained_models[model_key],
                imgs_train,
                latents_classes_train,
                max_samples=10000
            )
            
            mig_results[model_key] = {
                'beta': beta,
                'mig_score': mig_score,
                'mig_per_factor': mig_per_factor,
                'mi_matrix': mi_matrix
            }
        
        # Compare MIG scores
        print("\n" + "="*80)
        print("MIG SCORE COMPARISON")
        print("="*80)
        print(f"{'Model':<20} {'MIG Score':<15} {'Status'}")
        print("-"*80)
        for beta in BETA_VALUES:
            model_key = f'beta_{beta}'
            mig = mig_results[model_key]['mig_score']
            
            if mig < 0.2:
                status = "Poor disentanglement"
            elif mig < 0.4:
                status = "Moderate disentanglement"
            else:
                status = "Good disentanglement"
            
            print(f"β={beta:<18} {mig:<15.4f} {status}")
        
        print("\nExpected trend: Higher β → Higher MIG (better disentanglement)")
        
        # ====================================================================
        # STEP 6: Compare reconstructions side-by-side
        # ====================================================================
        print(f"\n[STEP 6/6] Generating Comparison Visualizations")
        print("="*80)
        
        # Get sample batch
        sample_data = next(iter(val_loader))[:8].to(device)
        
        fig, axes = plt.subplots(len(BETA_VALUES) + 1, 8, figsize=(16, 2*(len(BETA_VALUES)+1)))
        
        # Ensure axes is 2D
        if len(axes.shape) == 1:
            axes = axes.reshape(-1, 8)
        
        with torch.no_grad():
            reconstructions = {}
            for beta in BETA_VALUES:
                model_key = f'beta_{beta}'
                recon, _, _ = trained_models[model_key](sample_data)
                reconstructions[beta] = recon
        
        for i in range(8):
            # Original
            axes[0, i].imshow(sample_data[i, 0].cpu(), cmap='gray')
            axes[0, i].axis('off')
            if i == 0:
                axes[0, i].set_ylabel('Original', fontsize=10, fontweight='bold')
            
            # Reconstructions for each beta
            for idx, beta in enumerate(BETA_VALUES, 1):
                axes[idx, i].imshow(reconstructions[beta][i, 0].cpu(), cmap='gray')
                axes[idx, i].axis('off')
                if i == 0:
                    axes[idx, i].set_ylabel(f'β={beta}', fontsize=10, fontweight='bold')
        
        plt.suptitle('Reconstruction Comparison: Different β Values', 
                    fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.savefig('reconstruction_comparison_all.png', dpi=300, bbox_inches='tight')
        plt.show()
        
        print("✓ Comparison visualization saved: reconstruction_comparison_all.png")
        
        # PCA visualizations
        for beta in BETA_VALUES:
            model_key = f'beta_{beta}'
            print(f"\nGenerating PCA visualization for β={beta}...")
            pca, latent_2d = visualize_latent_space_2d(
                trained_models[model_key],
                val_loader,
                latents_classes_train,
                save_path=f'pca_beta{beta}.png'
            )
        
        # ====================================================================
        # FINAL SUMMARY
        # ====================================================================
        print("\n" + "="*80)
        print("TRAINING COMPLETED SUCCESSFULLY!")
        print("="*80)
        print("\nGenerated files:")
        print("  Models:")
        for beta in BETA_VALUES:
            print(f"    - vae_beta{beta}.pth")
        print("\n  Training history plots:")
        for beta in BETA_VALUES:
            print(f"    - training_history_beta{beta}.png")
        print("\n  Reconstruction visualizations:")
        for beta in BETA_VALUES:
            print(f"    - reconstructions_beta{beta}.png")
        print("    - reconstruction_comparison_all.png")
        print("\n  PCA visualizations:")
        for beta in BETA_VALUES:
            print(f"    - pca_beta{beta}.png")
        print("\n  Network diagrams:")
        print("    - bayesian_network.png")
        print("    - given_bayesian_network.png")
        print("    - markov_network.png")
        print("    - seven_node_markov_network.png")
        
        print("\n" + "="*80)
        print("SUMMARY OF RESULTS")
        print("="*80)
        
        for beta in BETA_VALUES:
            model_key = f'beta_{beta}'
            history = training_histories[model_key]
            mig_data = mig_results[model_key]
            
            print(f"\nβ={beta}:")
            print(f"  Final losses:")
            print(f"    Total: {history['val_loss'][-1]:.4f}")
            print(f"    Reconstruction: {history['val_recon'][-1]:.4f}")
            print(f"    KL Divergence: {history['val_kl'][-1]:.4f}")
            print(f"  MIG Score: {mig_data['mig_score']:.4f}")
        
        print("\n" + "="*80)
        print("All experiments completed! You can now analyze the results.")
        print("="*80)

else:
    print("\n⚠ Training skipped. Set TRAIN_MODELS=True to run experiments.")

## Additional: Question 1 - Part 4 (Variational Inference)

---

## Variational Inference: Theoretical Problem

This problem demonstrates how to derive optimal variational parameters for a given model.

### Problem Setup

We have:
- **Prior**: p(z) = e^(-z) for z > 0 (Exponential distribution)
- **Likelihood**: p(x|z) = z·e^(-zx) for x > 0
- **Variational family**: q(z) = θ²z·e^(-θz) (Gamma distribution with α=2, β=θ)

### Goal

Find θ* that maximizes the ELBO (Evidence Lower Bound):
$$\text{ELBO}(θ) = \mathbb{E}_{q(z)}[\log p(x,z)] - \mathbb{E}_{q(z)}[\log q(z)]$$

### Solution Strategy

1. **Expand ELBO** into reconstruction and KL terms
2. **Use Gamma distribution properties** to compute expectations
3. **Take derivative** with respect to θ
4. **Set to zero** and solve for optimal θ*

The solution shows that variational inference can find closed-form optimal parameters for simple models, which serves as a foundation for more complex approximate inference in deep generative models like VAE.

In [ ]:
print("Question 1 - Part 4: Variational Inference")
print("="*80)

print("\nGiven distributions:")
print("  p(z) = e^(-z), z > 0")
print("  p(x|z) = z * e^(-zx), x > 0")
print("  q(z) = θ² z e^(-θz), z > 0")
print("  E_q[z] = 2/θ")

print("\n" + "-"*80)
print("Solution:")
print("-"*80)

print("""
The ELBO (Evidence Lower Bound) is:
    L(θ) = E_q[log p(x,z)] - E_q[log q(z)]
         = E_q[log p(x|z)] + E_q[log p(z)] - E_q[log q(z)]

Step 1: Compute E_q[log p(x|z)]
    log p(x|z) = log(z) - zx
    E_q[log p(x|z)] = E_q[log z] - x E_q[z]
    
For Gamma distribution q(z) = θ² z e^(-θz):
    E_q[z] = 2/θ
    E_q[log z] = ψ(2) - log(θ)
    
    E_q[log p(x|z)] = ψ(2) - log(θ) - 2x/θ

Step 2: Compute E_q[log p(z)]
    log p(z) = -z
    E_q[log p(z)] = -E_q[z] = -2/θ

Step 3: Compute E_q[log q(z)]
    log q(z) = 2log(θ) + log(z) - θz
    E_q[log q(z)] = 2log(θ) + E_q[log z] - θE_q[z]
                  = 2log(θ) + ψ(2) - log(θ) - 2
                  = log(θ) + ψ(2) - 2

Step 4: Combine and find ELBO
    L(θ) = [ψ(2) - log(θ) - 2x/θ] + [-2/θ] - [log(θ) + ψ(2) - 2]
         = -2log(θ) - 2x/θ - 2/θ + 2
         = -2log(θ) - 2(x+1)/θ + 2

Step 5: Maximize ELBO by taking derivative
    dL/dθ = -2/θ + 2(x+1)/θ² = 0
    
    Solving: -2/θ + 2(x+1)/θ² = 0
             -2θ + 2(x+1) = 0
             θ = x + 1

Therefore, the optimal parameter is:
    θ* = x + 1
""")

def elbo_function(theta, x):
    import scipy.special as sp
    psi_2 = sp.digamma(2)
    return -2*np.log(theta) - 2*(x+1)/theta + 2

x_example = 1.0
theta_range = np.linspace(0.1, 5, 100)
elbo_values = [elbo_function(t, x_example) for t in theta_range]

plt.figure(figsize=(10, 6))
plt.plot(theta_range, elbo_values, 'b-', linewidth=2, label='ELBO(θ)')
plt.axvline(x=x_example + 1, color='r', linestyle='--', linewidth=2, 
            label=f'Optimal θ* = {x_example + 1}')
plt.xlabel('θ', fontsize=12)
plt.ylabel('ELBO', fontsize=12)
plt.title(f'ELBO as Function of θ (x = {x_example})', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('variational_inference_elbo.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\nFor x = {x_example}, optimal θ* = {x_example + 1}")
print(f"Maximum ELBO = {elbo_function(x_example + 1, x_example):.4f}")


### Mathematical Derivation and Visualization

This cell provides:
1. **Step-by-step derivation** of the optimal θ parameter
2. **ELBO visualization** as a function of θ
3. **Verification** that θ* = x + 1 maximizes ELBO

**Key Insights**:
- The optimal parameter depends on the observed data (x)
- The ELBO is concave in θ (has unique maximum)
- The solution θ* = x + 1 can be verified by plotting

This example bridges theory and practice:
- **Theory**: Variational inference principles
- **Practice**: Numerical optimization and visualization
- **Connection to VAE**: Similar principles but with neural networks instead of closed-form solutions

---

## Summary and Checklist

### Question 1: Probabilistic Graphical Models ✓
- [x] **Part 1**: Bayesian Network (Disease Model)
  - [x] Drew Bayesian network structure
  - [x] Derived joint probability distribution
  - [x] Analyzed conditional independence statements
  
- [x] **Part 2**: Given Bayesian/Markov Network
  - [x] Joint probability for Bayesian network
  - [x] Markov blanket of T
  - [x] Perfect I-map analysis
  - [x] Chordality check
  - [x] Maximal cliques identification
  - [x] Joint probability based on cliques

- [x] **Part 3**: Markov Network Properties
  - [x] Maximal cliques
  - [x] Conditional independence verification
  - [x] Effect of changing potential functions

- [x] **Part 4**: Variational Inference
  - [x] Derived optimal θ parameter
  - [x] Plotted ELBO function

### Question 2: Variational Autoencoders ✓
- [x] **Theoretical**:
  - [x] Why not maximize log-likelihood directly
  - [x] dSprites dataset description
  - [x] Reparameterization trick explanation
  
- [x] **Implementation**:
  - [x] VAE architecture (Encoder + Decoder)
  - [x] Training loop with loss tracking
  - [x] Reconstruction visualization
  
- [x] **β-VAE**:
  - [x] Explanation of β-VAE benefits
  - [x] Training with multiple β values
  - [x] Comparison of results
  
- [x] **Evaluation**:
  - [x] MIG metric implementation
  - [x] Disentanglement evaluation
  - [x] PCA visualization
  
- [x] **VAE Variants**:
  - [x] VQ-VAE explanation
  - [x] VampPrior explanation
  - [x] SC-VAE explanation

---

## Instructions to Run

1. **Download dSprites dataset**:
   ```python
   wget https://github.com/deepmind/dsprites-dataset/raw/master/dsprites_ndarray_co1sh3sc6or40x32y32_64x64.npz
   ```

2. **Uncomment training code** in the appropriate cells

3. **Run cells sequentially** to:
   - Load and visualize data
   - Train VAE models with different β
   - Evaluate disentanglement
   - Generate visualizations

4. **All outputs** (plots, models) will be saved automatically

---

## Key Results to Report

1. **Training curves** for β = 1, 2, 5
2. **Reconstruction quality** comparison
3. **MIG scores** for each model
4. **PCA visualizations** of latent space
5. **Analysis** of β effect on disentanglement

---

**Good luck with your homework! 🎓**

---

## Summary and Implementation Guide

### What This Notebook Covers

**Question 1: Probabilistic Graphical Models** ✓
- Bayesian networks with conditional independence
- Markov networks and clique factorization
- Graph analysis (moralization, chordality)
- Variational inference derivations

**Question 2: Variational Autoencoders** ✓
- Complete VAE implementation (encoder + decoder)
- Training pipeline with monitoring
- β-VAE for disentanglement
- MIG metric for quantitative evaluation
- Advanced variants (VQ-VAE, VampPrior, SC-VAE)

### Key Results to Obtain

**Visualizations**:
1. Bayesian/Markov network diagrams
2. Training curves (loss, reconstruction, KL)
3. Original vs reconstructed images
4. Latent space PCA projections
5. Mutual information matrices

**Metrics**:
1. Final training/validation losses
2. MIG scores for different β values
3. Per-factor disentanglement scores

**Analysis**:
1. Effect of β on reconstruction quality
2. Effect of β on disentanglement (MIG)
3. Which latent dimensions capture which factors

### How to Run This Notebook

**Prerequisites**:
```bash
pip install torch torchvision numpy matplotlib networkx scikit-learn tqdm scipy
```

**Download Dataset**:
```bash
wget https://github.com/deepmind/dsprites-dataset/raw/master/dsprites_ndarray_co1sh3sc6or40x32y32_64x64.npz
```

**Execution Order**:
1. Run all cells sequentially
2. Uncomment training code in training cells
3. Monitor progress with tqdm bars
4. Visualizations saved automatically

**GPU Acceleration**:
- Code automatically detects CUDA availability
- Training 50 epochs: ~30 minutes (GPU) vs ~3 hours (CPU)

### Expected Outcomes

**Quantitative**:
- Standard VAE (β=1): MIG ≈ 0.1-0.2
- β-VAE (β=2): MIG ≈ 0.2-0.3
- β-VAE (β=5): MIG ≈ 0.3-0.5

**Qualitative**:
- Higher β: blurrier reconstructions but better factor separation
- Lower β: sharper reconstructions but entangled representations
- Latent traversals: smooth transitions in factor space

### Troubleshooting

**Common Issues**:
1. **Out of memory**: Reduce batch_size or subset_size
2. **Slow training**: Ensure GPU is available (`torch.cuda.is_available()`)
3. **NaN loss**: Lower learning rate or check data normalization
4. **Poor MIG**: Train longer or increase β

**Debugging Tips**:
- Print model output shapes at each layer
- Visualize reconstructions early (epoch 5-10)
- Check KL doesn't collapse to zero
- Monitor gradient norms

---

**This notebook provides a complete, production-ready implementation of VAE and β-VAE with proper evaluation metrics. Good luck! 🚀**